# Evaluation of Synthetic Data Generation Methods for Tabular Health Data

## Models Evaluated

In this study we evaluate both probabilistic and generative synthetic data generation methods on tabular health data,
using implementations from **Synthetic Data Vault (SDV)** and **synthcity** Python libraries.

For each model we also try a differentially private (DP) version and later evaluate both.

### Probabilistic

1. **Gaussian Copulas**
    - `GaussianCopulaSynthesizer`
    - `DPGCSynthesizer` (DP version)
2. **Bayesian Networks**
    - `BayesianNetworkPlugin`
    - `PrivBayes` (DP version)

### Generative

1. **Generative Adversarial Networks**
    - `CTGAN`
    - DP variants: `DP-GAN`, `PATEGAN`, `AdsGAN`
2. **Variational Auto-Encoders**
    - `TVAE`, `RTVAE`
    - DP variants
3. **Normalizing Flows**
4. **Autoregressive / Tree-Based Models**
    - ARF (Adversarial Random Forests)
6. **Diffusion Models**
    - `TabDDPM` (DP)
    - `TableDiffusion` (DP)

## Data

A public clinical dataset **Heart Disease** from **UC Irvine Machine Learning Repository** is used in this study.

In [1]:
from ucimlrepo import fetch_ucirepo

heart_disease = fetch_ucirepo(id=45)

X = heart_disease.data.features
y = heart_disease.data.targets

X

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal
0,63,1,1,145,233,1,2,150,0,2.3,3,0.0,6.0
1,67,1,4,160,286,0,2,108,1,1.5,2,3.0,3.0
2,67,1,4,120,229,0,2,129,1,2.6,2,2.0,7.0
3,37,1,3,130,250,0,0,187,0,3.5,3,0.0,3.0
4,41,0,2,130,204,0,2,172,0,1.4,1,0.0,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
298,45,1,1,110,264,0,0,132,0,1.2,2,0.0,7.0
299,68,1,4,144,193,1,0,141,0,3.4,2,2.0,7.0
300,57,1,4,130,131,0,0,115,1,1.2,2,1.0,7.0
301,57,0,2,130,236,0,2,174,0,0.0,2,1.0,3.0


# Generative Models

## Generative Adversarial Network (GAN)

In this section we use GAN models to generate synthetic datasets from our original data.

### General Idea

There are two components - generator and discriminator. Generator creates a realistic synthetic row of tabular data with random noise applied. Discriminator acts as a binary classifier deciding whether the provided row is real or fake (0-1), taking either real or synthetic row as an input.

`[insert GAN loss function here]`

Discriminator (D) wants to classify correctly real samples (high probability) and fake samples (low probability).

Generator (G) wants to make Discrimator (D) classify fake samples as real.

### GAN on Tabular Data

Conditional Tabular Generative Adversarial Network (CTGAN) can explicitly condition the data generation process on specific categorical variables.

A typical CTGAN training loop:

1. Sample real batch from dataset.
2. Sample noise z.
3. Generator creates synthetic samples G(z).
4. Discriminator trains on:
    - real samples → label 1
    - fake samples → label 0
5. Generator trains to fool the discriminator:
    - update G so D(G(z)) → 1
6. Apply conditional constraints (CTGAN)
7. Repeat until convergence.

In [4]:
from synthcity.plugins import Plugins

Plugins().list()

[2025-11-17T22:57:43.284056+0000][11502][CRITICAL] module disabled: /workspaces/synth_tab_health_data/.venv/lib/python3.12/site-packages/synthcity/plugins/generic/plugin_goggle.py


['aim',
 'image_adsgan',
 'tvae',
 'image_cgan',
 'ctgan',
 'nflow',
 'great',
 'fflows',
 'pategan',
 'timevae',
 'survival_ctgan',
 'bayesian_network',
 'radialgan',
 'decaf',
 'marginal_distributions',
 'survival_gan',
 'arf',
 'ddpm',
 'privbayes',
 'uniform_sampler',
 'timegan',
 'dummy_sampler',
 'survival_nflow',
 'dpgan',
 'adsgan',
 'survae',
 'rtvae']

Create a train and test data loaders for **synthcity**.

In [5]:
from synthcity.plugins.core.dataloader import GenericDataLoader
X["target"] = y
loader = GenericDataLoader(
    X,
    target_column="targer",
)
train_loader, test_loader = loader.train(), loader.test()

Load the CTGAN plugin class.

In [6]:
plugin_cls = type(Plugins().get("ctgan"))
plugin_cls
# plugin = Plugins().get("ctgan", n_iter = 100)

[2025-11-17T22:57:52.339506+0000][11502][CRITICAL] module disabled: /workspaces/synth_tab_health_data/.venv/lib/python3.12/site-packages/synthcity/plugins/generic/plugin_goggle.py


synthcity.plugins.generic.plugin_ctgan.CTGANPlugin

### CTGAN plugin in synthcity

Most parameters of the plugin describe architectures or training loop of Generator (G) and Discriminator (D) neural networks.

#### Generator parameters

- `generator_n_layers_hidden`: how many hidden layers the generator neural network has (excluding input/out). More layers -> more capacity, but slower + risk of overfitting.
- `generator_n_units_hidden`: number of neurons per hidden layer. Controls generator model size.
- `generator_nonlin`: activation function used between layers:
    - `leaky_relu` - default, stable for GANs
    - `relu` - simpler
    - `elu` / `selu` - sometimes help with continuous skewed data
- `generator_dropout`: fraction of neurons randomly dropped during training. Prevents overfitting. Usually 0 for tabular GANs.
- `n_iter`: total number of training iterations (epochs). Higher -> better model (until overfitting), longer training.

#### Discriminator parameters

- `discriminator_n_layers_hidden`: how many hidden layers the discriminator has.
- `discriminator_n_units_hidden`: number of neurons per layer
- `discriminator_nonlin`: activation function for the discriminator.
- `discriminator_dropout`: dropout regularization for the discriminator.
- `discriminator_n_inter`: number of discriminator update steps per cycle.

#### Optimization and training

- `lr`: learning rate for both generator and discriminator
- `weight_decay`: L2 regularization on weights, which prevents exploding weights (and overfitting). Adds penalty to the loss function proportional to the sum of squares of the model's weights.
- `batch_size`: mini-batch size used during training. For a small dataset like in our case (303 rows), values 32-128 work good.
- `random_state`: random seed for reproducibility.
- `clipping_value`: gradient clipping threshold.

#### Preprocessing / encoding

- `encoder_max_clusters`: continuous variables are encoded using mode-specific normalization. This parameter controls the max number of mixture components (clusters) used. More clusters = better modeling of multimodal distributions, slower training.

To automatically tune hyperparameters in `synthcity`, we use hyperparameter optimization (HPO) algorithms such as Tree-structured Parzen estimators (TPE), Bayesian optimization, and genetic programming.

We will use `optuna` HPO library, which implements TPE, to tune hyperparameters.

Display the hyperparameter space.

In [15]:
plugin_cls.hyperparameter_space()

[IntegerDistribution(name='generator_n_layers_hidden', data=None, random_state=None, sampling_strategy='marginal', marginal_distribution=None, low=1, high=4, step=1),
 IntegerDistribution(name='generator_n_units_hidden', data=None, random_state=None, sampling_strategy='marginal', marginal_distribution=None, low=50, high=150, step=50),
 CategoricalDistribution(name='generator_nonlin', data=None, random_state=None, sampling_strategy='marginal', marginal_distribution=elu           0.25
 leaky_relu    0.25
 relu          0.25
 tanh          0.25
 dtype: float64, choices=['elu', 'leaky_relu', 'relu', 'tanh']),
 IntegerDistribution(name='n_iter', data=None, random_state=None, sampling_strategy='marginal', marginal_distribution=None, low=100, high=1000, step=100),
 FloatDistribution(name='generator_dropout', data=None, random_state=None, sampling_strategy='marginal', marginal_distribution=None, low=0.0, high=0.2),
 IntegerDistribution(name='discriminator_n_layers_hidden', data=None, random_st

Use a trial to suggest a set of hyperparameters.

In [13]:
import optuna
from synthcity.utils.optuna_sample import suggest_all

trial = optuna.create_study().ask()
params = suggest_all(trial, plugin_cls.hyperparameter_space())
# params["n_iter"] = 100 # speed up
params

{'generator_n_layers_hidden': 2,
 'generator_n_units_hidden': 150,
 'generator_nonlin': 'leaky_relu',
 'n_iter': 400,
 'generator_dropout': 0.19716837418279443,
 'discriminator_n_layers_hidden': 3,
 'discriminator_n_units_hidden': 100,
 'discriminator_nonlin': 'elu',
 'discriminator_n_iter': 1,
 'discriminator_dropout': 0.03515249306602961,
 'lr': 0.0002,
 'weight_decay': 0.0001,
 'batch_size': 500,
 'encoder_max_clusters': 14}

Evaluate the plugin with the suggested hyperparameters.

In [ ]:
from synthcity.benchmark import Benchmarks

plugin = plugin_cls(**params).fit(train_loader)
report = Benchmarks.evaluate(
    [("trial", "ctgan", params)],
    train_loader, # Benchmarks.evaluate will split out a validation set
    repeats=1,
    # metrics={"detection": ["detection_xgb"]}
)
report["trial"]

100%|██████████| 400/400 [00:16<00:00, 24.37it/s]
[2025-11-17T23:06:47.853578+0000][11502][CRITICAL] module disabled: /workspaces/synth_tab_health_data/.venv/lib/python3.12/site-packages/synthcity/plugins/generic/plugin_goggle.py
100%|█████████▉| 399/400 [00:14<00:00, 26.74it/s]


,min,max,mean,stddev,median,iqr,rounds,errors,durations,direction
detection.detection_xgb.mean,0.99,0.99,0.99,0.0,0.99,0.0,1,0,0.09,minimize


Create an Optuna study and optimize the hyperparameters.

In [15]:
import pandas as pd

def objective(trial: optuna.Trial):
    hp_space = Plugins().get("ctgan").hyperparameter_space()
    # hp_space[0].high = 100 # speed up for now
    params = suggest_all(trial, hp_space)
    ID = f"trial_{trial.number}"
    try:
        report = Benchmarks.evaluate(
            [(ID, "ctgan", params)],
            train_loader,
            repeats=1,
            metrics={"detection": ["detection_xgb"]},
        )
    except Exception as e: # invalid set of params
        print(f"{type(e).__name__}: {e}")
        print(params)
        raise optuna.TrialPruned()
    score = report[ID].query('direction == "minimize"')["mean"].mean()
    # df = report[ID]
    # minimize metrics
    # minimize_vals = df.query('direction == "minimize"')["mean"]
    # maximize metrics → turn into minimize form by negating
    # maximize_vals = -df.query('direction == "maximize"')["mean"]
    # final scalar objective
    # score = pd.concat([minimize_vals, maximize_vals]).mean()

    return score

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=2)
study.best_params

[2025-11-17T23:10:51.444197+0000][11502][CRITICAL] module disabled: /workspaces/synth_tab_health_data/.venv/lib/python3.12/site-packages/synthcity/plugins/generic/plugin_goggle.py
[2025-11-17T23:10:51.455841+0000][11502][CRITICAL] module disabled: /workspaces/synth_tab_health_data/.venv/lib/python3.12/site-packages/synthcity/plugins/generic/plugin_goggle.py
100%|██████████| 300/300 [00:29<00:00, 10.25it/s]
[2025-11-17T23:11:21.865539+0000][11502][CRITICAL] module disabled: /workspaces/synth_tab_health_data/.venv/lib/python3.12/site-packages/synthcity/plugins/generic/plugin_goggle.py
[2025-11-17T23:11:21.873012+0000][11502][CRITICAL] module disabled: /workspaces/synth_tab_health_data/.venv/lib/python3.12/site-packages/synthcity/plugins/generic/plugin_goggle.py
 78%|███████▊  | 549/700 [00:19<00:05, 28.48it/s]


{'generator_n_layers_hidden': 3,
 'generator_n_units_hidden': 150,
 'generator_nonlin': 'tanh',
 'n_iter': 300,
 'generator_dropout': 0.05035603815930367,
 'discriminator_n_layers_hidden': 2,
 'discriminator_n_units_hidden': 50,
 'discriminator_nonlin': 'leaky_relu',
 'discriminator_n_iter': 2,
 'discriminator_dropout': 0.19461072590174788,
 'lr': 0.001,
 'weight_decay': 0.0001,
 'batch_size': 100,
 'encoder_max_clusters': 15}

Visualize the study.

In [ ]:
from optuna.visualization import (plot_contour,
                                  plot_edf,
                                  plot_optimization_history,
                                  plot_parallel_coordinate,
                                  plot_param_importances,
                                  plot_slice)

plot_optimization_history(study)

In [ ]:
# visualize high-dimensional parameter relationships
plot_parallel_coordinate(study)

In [ ]:
# visualize hyperparameter relationships
fig = plot_contour(study, params=["batch_size", "lr", "generator_n_layers_hidden", "discriminator_n_layers_hidden"])
fig.update_layout(width=800, height=800)

In [ ]:
# visualize individual hyperparameters as slice plot
plot_slice(study)

In [ ]:
# visualize parameter importances
plot_param_importances(study)

In [ ]:
# learn which hyperparameters are affecting the trial duration with hyperparameter importance
optuna.visualization.plot_param_importances(
    study, target=lambda t: t.duration.total_seconds(), target_name="duration"
)

In [ ]:
# visualize empirical distribution function of the objective
plot_edf(study)

Test performance of the optimized plugin.

In [16]:
best_params = study.best_params
report = Benchmarks.evaluate(
    [("test", "ctgan", best_params)],
    train_loader,
    test_loader,
    repeats=1,
    # metrics={"detection": ["detection_mlp", "detection_xgb"]},
)
Benchmarks.print(report)

[2025-11-17T23:12:27.090351+0000][11502][CRITICAL] module disabled: /workspaces/synth_tab_health_data/.venv/lib/python3.12/site-packages/synthcity/plugins/generic/plugin_goggle.py
100%|██████████| 300/300 [00:28<00:00, 10.70it/s]



Plugin : test


,min,max,mean,stddev,median,iqr,rounds,errors,durations
sanity.data_mismatch.score,0.000000,0.000000,0.000000,0.0,0.000000,0.0,1,0,0.00
sanity.common_rows_proportion.score,0.000000,0.000000,0.000000,0.0,0.000000,0.0,1,0,0.01
sanity.nearest_syn_neighbor_distance.mean,0.082438,0.082438,0.082438,0.0,0.082438,0.0,1,0,0.00
sanity.close_values_probability.score,0.967213,0.967213,0.967213,0.0,0.967213,0.0,1,0,0.00
sanity.distant_values_probability.score,0.016393,0.016393,0.016393,0.0,0.016393,0.0,1,0,0.00
stats.jensenshannon_dist.marginal,0.018719,0.018719,0.018719,0.0,0.018719,0.0,1,0,0.07
stats.chi_squared_test.marginal,0.754828,0.754828,0.754828,0.0,0.754828,0.0,1,0,0.02
stats.inv_kl_divergence.marginal,0.858335,0.858335,0.858335,0.0,0.858335,0.0,1,0,0.01
stats.ks_test.marginal,0.860656,0.860656,0.860656,0.0,0.860656,0.0,1,0,0.01
stats.max_mean_discrepancy.joint,0.032787,0.032787,0.032787,0.0,0.032787,0.0,1,0,0.01


In [49]:
best_params

{'generator_n_layers_hidden': 4,
 'generator_n_units_hidden': 50,
 'generator_nonlin': 'relu',
 'n_iter': 400,
 'generator_dropout': 0.12026413697886398,
 'discriminator_n_layers_hidden': 4,
 'discriminator_n_units_hidden': 150,
 'discriminator_nonlin': 'tanh',
 'discriminator_n_iter': 3,
 'discriminator_dropout': 0.15977502374920438,
 'lr': 0.0001,
 'weight_decay': 0.001,
 'batch_size': 100,
 'encoder_max_clusters': 18}

In [50]:
plugin.generate(50, **best_params)

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,59,1,1,134,225,1,0,139,0,0.045249,2,3.0,3.0,2
1,60,1,4,153,211,0,2,149,0,0.001238,2,3.0,7.0,3
2,55,1,4,121,211,1,0,140,0,1.058687,1,0.0,3.0,2
3,60,0,2,137,220,1,2,173,1,0.967326,2,0.0,7.0,4
4,62,0,4,133,200,0,2,129,0,1.939072,2,NaN,3.0,2
5,60,1,4,129,196,1,2,144,0,0.000000,1,3.0,7.0,1
6,57,1,3,147,202,0,2,144,0,2.166824,1,2.0,7.0,4
7,60,1,4,126,191,0,2,160,0,1.274699,3,1.0,7.0,4
8,63,1,1,132,203,0,2,131,0,1.739298,3,2.0,3.0,2
9,60,1,4,127,186,0,2,138,0,2.055985,2,2.0,7.0,1
